# DSA210 Milestone 2 - Dilay Arkın
## Apply Machine Learning Methods on the Dataset

Applied supervised machine learning methods to analyze the relationship between AI usage and academic performance.

In [2]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("dataset.csv")
df.head()

Saving dataset.csv to dataset (2).csv


,date,ai_usage_minutes,purpose,num_interactions,study_duration_times,productivity,task_type,deadline_near,course_type,difficulty,performance_score
0,2026-03-07,45,learning,6,135,2,reading,no,qualitative,3,81
1,2026-03-08,80,assignment,10,175,3,coding,yes,quantitative,4,74
2,2026-03-09,35,quick_answer,5,115,3,writing,no,qualitative,2,80
3,2026-03-10,95,assignment,13,200,3,coding,yes,quantitative,5,71
4,2026-03-11,55,learning,7,145,4,reading,no,qualitative,3,83


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   date                  60 non-null     object
 1   ai_usage_minutes      60 non-null     int64 
 2   purpose               60 non-null     object
 3   num_interactions      60 non-null     int64 
 4   study_duration_times  60 non-null     int64 
 5   productivity          60 non-null     int64 
 6   task_type             60 non-null     object
 7   deadline_near         60 non-null     object
 8   course_type           60 non-null     object
 9   difficulty            60 non-null     int64 
 10  performance_score     60 non-null     int64 
dtypes: int64(6), object(5)
memory usage: 5.3+ KB


In [4]:
df.isnull().sum()

,0
date,0
ai_usage_minutes,0
purpose,0
num_interactions,0
study_duration_times,0
productivity,0
task_type,0
deadline_near,0
course_type,0
difficulty,0


The dataset contains 60 observations. No missing values were found, so no additional data cleaning was required.

In [5]:
df_encoded = pd.get_dummies(
    df,
    columns=["purpose", "task_type", "deadline_near", "course_type"],
    drop_first=True
)

df_encoded.head()

,date,ai_usage_minutes,num_interactions,study_duration_times,productivity,difficulty,performance_score,purpose_learning,purpose_quick_answer,task_type_reading,task_type_writing,deadline_near_yes,course_type_quantitative
0,2026-03-07,45,6,135,2,3,81,True,False,True,False,False,False
1,2026-03-08,80,10,175,3,4,74,False,False,False,False,True,True
2,2026-03-09,35,5,115,3,2,80,False,True,False,True,False,False
3,2026-03-10,95,13,200,3,5,71,False,False,False,False,True,True
4,2026-03-11,55,7,145,4,3,83,True,False,True,False,False,False


## Regression Task: Predicting Performance Score

In [6]:
X = df_encoded.drop(columns=["date", "performance_score"])
y = df_encoded["performance_score"]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

MAE: 1.2225093124635862
RMSE: 1.6107948558038545
R2: 0.9105931922187798


In [10]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=3)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("Tree R2:", r2_score(y_test, y_pred_tree))

Tree R2: 0.8852506784000991


In [11]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

print("KNN R2:", r2_score(y_test, y_pred_knn))

KNN R2: 0.9357932519741566


Linear Regression, Decision Tree, and KNN models were applied to predict performance scores.

The models achieved relatively high R² scores, indicating strong predictive performance on this dataset.

However, given the self-collected and relatively small dataset, these results may be overly optimistic and should be interpreted with caution.

The high performance may also suggest potential overfitting, meaning the models might not generalize well to new, unseen data.

Overall, while AI usage and study-related features appear to be useful predictors, academic performance is influenced by multiple factors beyond those captured in this dataset.

## Classification Task: Predicting High Productivity

In [12]:
df_encoded["high_productivity"] = df_encoded["productivity"].apply(lambda x: 1 if x >= 4 else 0)

In [13]:
X_clf = df_encoded.drop(columns=["date", "performance_score", "productivity", "high_productivity"])
y_clf = df_encoded["high_productivity"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

In [15]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.75
Precision: 0.4
Recall: 1.0
F1: 0.5714285714285714


In [17]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("Tree Accuracy:", accuracy_score(y_test, y_pred_tree))

Tree Accuracy: 0.8333333333333334


In [18]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

KNN Accuracy: 0.75


Classification models were applied to predict whether a student has high productivity.

The models achieved moderate accuracy in classification tasks.

However, the evaluation metrics show an imbalance between precision and recall. In particular, the model has high recall but relatively low precision, indicating that it tends to over-predict high productivity cases.

This suggests that while the model is good at identifying high productivity instances, it may also produce a higher number of false positives.

Therefore, the classification results should be interpreted cautiously, and further improvements with more balanced data or additional features may be needed.

## Conclusion

Overall, machine learning methods were successfully applied to analyze the relationship between AI usage and academic performance.

While the models show promising results, the limited size and synthetic nature of the dataset restrict the reliability and generalizability of the findings.

Future work could include collecting more data over a longer time period and incorporating additional variables to improve model performance and robustness.